# 04 — Knowledge Graph Construction

**Pipeline Step 4 of 5**

Build a BioCypher Micro-Clinical Knowledge Graph from 
Stabl-selected biomarkers and spatial cluster data.

### Inputs
- `data/processed/mouse_brain_preprocessed.h5ad`
- `cache/stabl_results_<hash>.pkl`
- `config/schema_config.yaml`

### Outputs
- `cache/micro_ckg.graphml`

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.spatial_pipeline import load_adata, run_stabl_cached, compute_clusters
from src.biocypher_adapter import build_micro_ckg, save_graph

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
CACHE_DIR = PROJECT_ROOT / "cache"

print("Imports ready.")

Imports ready.


## 4.1 Load Data & Stabl Results

In [2]:
adata = load_adata(DATA_PROCESSED / "mouse_brain_preprocessed.h5ad")

stabl_result = run_stabl_cached(
    adata,
    cache_dir=CACHE_DIR,
    dataset_name="squidpy",
    n_hvgs=2000,
    label_method="cluster",
    n_bootstraps=500,
)

print(f"\n{stabl_result['n_selected']} Stabl features loaded.")

  Loading dataset: /Users/shaunfchen/Documents/Repositories/spatial-microckg-agent/data/processed/mouse_brain_preprocessed.h5ad
  Shape: 2691 spots × 19652 genes
  Loading cached Stabl results: /Users/shaunfchen/Documents/Repositories/spatial-microckg-agent/cache/stabl_results_eba9686a7ee7.pkl

41 Stabl features loaded.


## 4.2 Compute Leiden Clusters

The KG needs Leiden clusters to create CellType and AnatomicalEntity nodes.

In [3]:
adata = compute_clusters(adata, n_hvgs=2000)
print(f"\nLeiden clusters: {adata.obs['leiden'].nunique()}")
print(adata.obs["leiden"].value_counts().sort_index())

  Selected 2000 highly variable genes (requested 2000)


/Users/shaunfchen/.local/share/uv/python/cpython-3.11.15-macos-aarch64-none/lib/python3.11/functools.py:909: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)


/Users/shaunfchen/Documents/Repositories/spatial-microckg-agent/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  Leiden clustering: 17 clusters

Leiden clusters: 17
leiden
0      64
1     198
2     194
3     103
4     114
5     220
6     291
7     337
8     124
9     130
10    207
11    205
12     36
13    259
14     48
15     70
16     91
Name: count, dtype: int64


## 4.3 Build Micro-CKG

- **Gene** nodes — Stabl-selected markers with stability scores
- **CellType** nodes — Leiden clusters
- **AnatomicalEntity** nodes — Brain regions
- Edges: Gene→CellType, Gene→Region, CellType→Region

In [4]:
schema_path = PROJECT_ROOT / "config" / "schema_config.yaml"

graph = build_micro_ckg(
    stabl_result=stabl_result,
    adata=adata,
    schema_path=schema_path,
)

print(f"\nMicro-CKG:")
print(f"  Nodes: {graph.number_of_nodes()}")
print(f"  Edges: {graph.number_of_edges()}")

  Building Micro-CKG...


  Micro-CKG: 66 nodes (41 genes, 17 cell types, 8 regions)
  Micro-CKG: 988 edges

Micro-CKG:
  Nodes: 66
  Edges: 988


## 4.4 Save Graph

In [5]:
graph_path = save_graph(graph, CACHE_DIR / "micro_ckg.graphml")
print(f"\nPersisted: {graph_path}")
print(f"File size: {graph_path.stat().st_size / 1e3:.1f} KB")

  Graph saved to /Users/shaunfchen/Documents/Repositories/spatial-microckg-agent/cache/micro_ckg.graphml

Persisted: /Users/shaunfchen/Documents/Repositories/spatial-microckg-agent/cache/micro_ckg.graphml
File size: 269.9 KB
